# Final_03_user.ipynb 步驟目錄
---
| Step | 內容 | 使用資料 | 輸出檔案 |
|---:|---|---|---|
| 1 | 讀取道路災前 / 災後旅行時間資料 | `output/road_grid_travel_time_pre_post.geojson` | `road_travel` |
| 2 | 彙整每個 grid 的平均道路旅行時間增加率 | `road_travel`, `output/Taipei_grid_full_risk.geojson` | `grid_travel_increase` |
| 3 | 建立可點擊的 grid 旅行時間增加率地圖 | `grid_travel_increase`, `data/TOWN_MOI_1140318/TOWN_MOI_1140318.shp` | `output/Taipei_grid_travel_time_increase_rate_map.html` |
| 4 | 輸出 grid 層級旅行時間增加率資料 | `grid_travel_increase` | `output/Taipei_grid_travel_time_increase_rate.geojson`, `output/Taipei_grid_travel_time_increase_rate.csv` |
| 5 | 比較路網密度、地形條件與旅行時間增加 | `road_travel`, `output/Taipei_grid_full_risk.geojson` | `grid_compare`, `comparison_summary` |
| 6 | 輸出路網密度 / 地形比較結果 | `grid_compare`, `comparison_summary` | `output/Taipei_grid_density_terrain_travel_time_comparison.geojson`, `output/Taipei_grid_density_terrain_travel_time_comparison.csv`, `output/Taipei_density_terrain_comparison_summary.csv` |

### Commander 資料讀取與路網準備
---
本段程式讀取第三階段指揮官模式所需的資料，並建立災害情境下可用於路徑規劃的道路 graph。

| 變數名稱 | 來源資料 | 說明 |
|---|---|---|
| `G` | `output/taipei_drive.graphml` | 臺北市原始道路 graph，並轉換為 `EPSG:3826` |
| `road_traveling` | `output/road_grid_traveling.csv` | 每段 road-grid 的災後車速、通行時間與道路狀態 |
| `road_grid_segments` | `output/taipei_road_grid_segments.geojson` | 被 grid 切分後的道路 geometry，保留 `road_grid_id` 與原始 edge ID |
| `roads` | `road_grid_segments` + `road_traveling` | 合併道路 geometry 與災後通行屬性的完整道路資料 |
| `edge_cost` | `roads` 彙整結果 | 將 road-grid 分段通行時間彙整回原始 graph edge |
| `G_route` | `G` 移除封閉 / 缺資料道路後 | 災害情境下實際用於路徑規劃的道路 graph |
| `shelters` | `output/taipei_shelters_join_500m.geojson` | 臺北市避難所資料，並已連結最近道路 |
| `taipei_boundary` | `data/TOWN_MOI_1140318/TOWN_MOI_1140318.shp` | 臺北市各行政區邊界 |
| `boundary` | `taipei_boundary.dissolve()` | 臺北市整體行政邊界 |

其中，`commander_travel_time` 是本階段最重要的路徑成本欄位，代表災害情境下每條道路 edge 的通行時間。程式會將封閉道路或缺少通行時間的道路移除，產生 `G_route`，供後續避難所路線規劃使用。

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import geopandas as gpd
import osmnx as ox
import networkx as nx

TARGET_CRS = "EPSG:3826"
OUTPUT_DIR = Path("output")

road_travel_path = OUTPUT_DIR / "road_grid_travel_time_pre_post.geojson"

if not road_travel_path.exists():
    raise FileNotFoundError(f"Missing file: {road_travel_path}")

road_travel = gpd.read_file(road_travel_path).to_crs(TARGET_CRS)

print("road travel rows:", len(road_travel))
print("CRS:", road_travel.crs)
print("columns:")
print(road_travel.columns.tolist())

display(road_travel.head())

In [ ]:
import numpy as np
import pandas as pd
import geopandas as gpd
import folium
import branca.colormap as cm
from pathlib import Path

# ============================================================
# Grid-level average travel time increase rate clickable map
# ============================================================

OUTPUT_DIR = Path("output")
TARGET_CRS = "EPSG:3826"

# ===== 0) Read input data =====
if "road_travel" not in globals():
    road_travel = gpd.read_file(
        OUTPUT_DIR / "road_grid_travel_time_pre_post.geojson"
    ).to_crs(TARGET_CRS)

if "grid" not in globals():
    grid = gpd.read_file(
        OUTPUT_DIR / "Taipei_grid_full_risk.geojson"
    ).to_crs(TARGET_CRS)

grid_base = grid[["grid_id", "geometry"]].copy()

# ===== 1) Clean road travel data =====
road_grid = road_travel.copy()

road_grid["segment_length_m"] = pd.to_numeric(
    road_grid["segment_length_m"],
    errors="coerce",
)

road_grid["pre_travel_time_sec"] = pd.to_numeric(
    road_grid["pre_travel_time_sec"],
    errors="coerce",
)

road_grid["post_travel_time_sec"] = pd.to_numeric(
    road_grid["post_travel_time_sec"],
    errors="coerce",
)

road_grid["is_closed_post"] = (
    road_grid["is_closed_post"]
    .fillna(False)
    .astype(bool)
)

open_road_grid = road_grid[
    (~road_grid["is_closed_post"])
    & road_grid["pre_travel_time_sec"].notna()
    & road_grid["post_travel_time_sec"].notna()
    & (road_grid["pre_travel_time_sec"] > 0)
].copy()

closed_road_grid = road_grid[
    road_grid["is_closed_post"]
].copy()

# ===== 2) Grid-level travel time statistics =====
grid_time_stats = (
    open_road_grid
    .groupby("grid_id", as_index=False)
    .agg(
        road_segment_count=("road_grid_id", "count"),
        total_open_road_length_m=("segment_length_m", "sum"),
        pre_travel_time_sum_sec=("pre_travel_time_sec", "sum"),
        post_travel_time_sum_sec=("post_travel_time_sec", "sum"),
    )
)

grid_time_stats["pre_avg_travel_time_sec"] = (
    grid_time_stats["pre_travel_time_sum_sec"]
    / grid_time_stats["road_segment_count"]
)

grid_time_stats["post_avg_travel_time_sec"] = (
    grid_time_stats["post_travel_time_sum_sec"]
    / grid_time_stats["road_segment_count"]
)

grid_time_stats["pre_avg_travel_time_min"] = (
    grid_time_stats["pre_avg_travel_time_sec"] / 60
)

grid_time_stats["post_avg_travel_time_min"] = (
    grid_time_stats["post_avg_travel_time_sec"] / 60
)

grid_time_stats["avg_travel_time_increase_min"] = (
    grid_time_stats["post_avg_travel_time_min"]
    - grid_time_stats["pre_avg_travel_time_min"]
)

grid_time_stats["avg_travel_time_increase_rate_pct"] = (
    (
        grid_time_stats["post_travel_time_sum_sec"]
        / grid_time_stats["pre_travel_time_sum_sec"]
        - 1
    )
    * 100
)

# ===== 3) Closed-road statistics by grid =====
grid_total_length = (
    road_grid
    .groupby("grid_id", as_index=False)
    .agg(
        total_road_length_m=("segment_length_m", "sum"),
        total_segment_count=("road_grid_id", "count"),
    )
)

grid_closed_length = (
    closed_road_grid
    .groupby("grid_id", as_index=False)
    .agg(
        closed_road_length_m=("segment_length_m", "sum"),
        closed_segment_count=("road_grid_id", "count"),
    )
)

grid_time_stats = grid_time_stats.merge(
    grid_total_length,
    on="grid_id",
    how="outer",
)

grid_time_stats = grid_time_stats.merge(
    grid_closed_length,
    on="grid_id",
    how="left",
)

grid_time_stats["closed_road_length_m"] = (
    grid_time_stats["closed_road_length_m"]
    .fillna(0)
)

grid_time_stats["closed_segment_count"] = (
    grid_time_stats["closed_segment_count"]
    .fillna(0)
    .astype(int)
)

grid_time_stats["closed_road_length_ratio_pct"] = np.where(
    grid_time_stats["total_road_length_m"] > 0,
    grid_time_stats["closed_road_length_m"]
    / grid_time_stats["total_road_length_m"]
    * 100,
    np.nan,
)

# ===== 4) Join stats back to grid =====
grid_travel_increase = grid_base.merge(
    grid_time_stats,
    on="grid_id",
    how="left",
)

numeric_fill_cols = [
    "avg_travel_time_increase_rate_pct",
    "closed_road_length_ratio_pct",
    "closed_road_length_m",
    "closed_segment_count",
]

for col in numeric_fill_cols:
    if col in grid_travel_increase.columns:
        grid_travel_increase[col] = grid_travel_increase[col].fillna(0)

# ===== 5) Read Taipei administrative district boundary =====
town_path = Path("data/TOWN_MOI_1140318/TOWN_MOI_1140318.shp")
town = gpd.read_file(town_path)

town.columns = [str(c).strip().strip("'").strip('"') for c in town.columns]

if "COUNTYCODE" in town.columns:
    taipei_town = town[
        town["COUNTYCODE"].astype(str).str.contains("63000", na=False)
    ].copy()
elif "COUNTYNAME" in town.columns:
    taipei_town = town[
        town["COUNTYNAME"].astype(str).str.contains(
            "臺北市|台北市",
            na=False,
            regex=True,
        )
    ].copy()
else:
    taipei_town = town.copy()

taipei_town = taipei_town.to_crs(TARGET_CRS)

# ===== 6) Save grid result =====
grid_output_geojson = OUTPUT_DIR / "Taipei_grid_travel_time_increase_rate.geojson"
grid_output_csv = OUTPUT_DIR / "Taipei_grid_travel_time_increase_rate.csv"

grid_travel_increase.to_file(
    grid_output_geojson,
    driver="GeoJSON",
    encoding="utf-8",
)

grid_travel_increase.drop(columns="geometry").to_csv(
    grid_output_csv,
    index=False,
    encoding="utf-8-sig",
)

print("Saved:", grid_output_geojson)
print("Saved:", grid_output_csv)

# ===== 7) Prepare map data =====
grid_map = grid_travel_increase.copy()

grid_map["geometry"] = grid_map.geometry.simplify(
    tolerance=5,
    preserve_topology=True,
)

grid_wgs84 = grid_map.to_crs(epsg=4326)
town_wgs84 = taipei_town.to_crs(epsg=4326)

value_col = "avg_travel_time_increase_rate_pct"

values = (
    grid_wgs84[value_col]
    .replace([np.inf, -np.inf], np.nan)
    .dropna()
)

vmin = 0
vmax = max(values.quantile(0.98), 1) if len(values) > 0 else 1

cmap = cm.linear.YlOrRd_09.scale(vmin, vmax)
cmap.caption = "Average travel time increase rate by grid (%)"

def fmt_num(value, digits=2, suffix=""):
    if pd.isna(value) or not np.isfinite(value):
        return "N/A"
    return f"{value:.{digits}f}{suffix}"

def fmt_int(value):
    if pd.isna(value):
        return "N/A"
    return f"{int(value)}"

grid_wgs84["pre_avg_time_text"] = grid_wgs84["pre_avg_travel_time_min"].apply(
    lambda x: fmt_num(x, 3, " min")
)

grid_wgs84["post_avg_time_text"] = grid_wgs84["post_avg_travel_time_min"].apply(
    lambda x: fmt_num(x, 3, " min")
)

grid_wgs84["increase_min_text"] = grid_wgs84["avg_travel_time_increase_min"].apply(
    lambda x: fmt_num(x, 3, " min")
)

grid_wgs84["increase_rate_text"] = grid_wgs84[value_col].apply(
    lambda x: fmt_num(x, 2, "%")
)

grid_wgs84["road_segment_count_text"] = grid_wgs84["road_segment_count"].apply(fmt_int)

grid_wgs84["total_road_length_text"] = grid_wgs84["total_road_length_m"].apply(
    lambda x: fmt_num(x, 1, " m")
)

grid_wgs84["closed_ratio_text"] = grid_wgs84["closed_road_length_ratio_pct"].apply(
    lambda x: fmt_num(x, 2, "%")
)

grid_wgs84["closed_segment_count_text"] = grid_wgs84["closed_segment_count"].apply(fmt_int)

# ===== 8) Build map =====
center_gdf = gpd.GeoDataFrame(
    geometry=taipei_town.dissolve().geometry.centroid,
    crs=TARGET_CRS,
).to_crs(epsg=4326)

center = center_gdf.geometry.iloc[0]

m = folium.Map(
    location=[center.y, center.x],
    zoom_start=12,
    tiles="cartodbpositron",
    zoom_control=False,
)

# ===== 9) Title panel =====
overall_rate = (
    (
        open_road_grid["post_travel_time_sec"].sum()
        / open_road_grid["pre_travel_time_sec"].sum()
        - 1
    )
    * 100
    if open_road_grid["pre_travel_time_sec"].sum() > 0
    else np.nan
)

title_html = f"""
<div style="
    position: fixed;
    top: 18px;
    left: 18px;
    z-index: 9999;
    width: 360px;
    background: rgba(255, 255, 255, 0.94);
    border: 1px solid #cccccc;
    border-radius: 8px;
    padding: 14px;
    font-family: Arial, sans-serif;
    box-shadow: 0 4px 18px rgba(0,0,0,0.18);
">
    <div style="font-size: 18px; font-weight: 700; margin-bottom: 6px;">
        臺北市網格道路旅行時間增加率
    </div>
    <div style="font-size: 13px; color: #555; line-height: 1.45;">
        顏色表示每個網格內道路 segment 的平均旅行時間增加率。點擊任一網格可查看災前/災後平均旅行時間與詳細統計。
    </div>
    <hr style="margin: 10px 0;">
    <div style="font-size: 13px;">
        <b>全市平均旅行時間增加率：</b>{fmt_num(overall_rate, 2, "%")}<br>
        <b>網格數：</b>{len(grid_wgs84)}<br>
        <b>道路 segment 數：</b>{len(road_grid)}
    </div>
</div>
"""

m.get_root().html.add_child(folium.Element(title_html))

# ===== 10) Clickable grid layer =====
grid_fg = folium.FeatureGroup(
    name="Grid average travel time increase rate",
    show=True,
)

for _, row in grid_wgs84.iterrows():
    value = row[value_col]

    if pd.isna(value):
        fill_color = "#f0f0f0"
        fill_opacity = 0.15
    else:
        fill_color = cmap(float(value))
        fill_opacity = 0.68

    popup_html = f"""
    <div style="font-family: Arial, sans-serif; font-size: 13px; line-height: 1.5;">
        <div style="font-size: 15px; font-weight: 700; margin-bottom: 4px;">
            Grid ID: {row["grid_id"]}
        </div>
        <hr style="margin: 6px 0;">
        <b>災前平均旅行時間：</b>{row["pre_avg_time_text"]}<br>
        <b>災後平均旅行時間：</b>{row["post_avg_time_text"]}<br>
        <b>平均增加時間：</b>{row["increase_min_text"]}<br>
        <b>平均增加率：</b>{row["increase_rate_text"]}<br>
        <hr style="margin: 6px 0;">
        <b>Road segment 數量：</b>{row["road_segment_count_text"]}<br>
        <b>道路總長度：</b>{row["total_road_length_text"]}<br>
        <b>封閉道路比例：</b>{row["closed_ratio_text"]}<br>
        <b>封閉 segment 數：</b>{row["closed_segment_count_text"]}
    </div>
    """

    tooltip_html = f"""
    <b>Grid ID:</b> {row["grid_id"]}<br>
    <b>增加率:</b> {row["increase_rate_text"]}<br>
    <b>災前平均:</b> {row["pre_avg_time_text"]}<br>
    <b>災後平均:</b> {row["post_avg_time_text"]}
    """

    folium.GeoJson(
        row.geometry.__geo_interface__,
        style_function=lambda feature, fill_color=fill_color, fill_opacity=fill_opacity: {
            "fillColor": fill_color,
            "color": "#666666",
            "weight": 0.35,
            "fillOpacity": fill_opacity,
        },
        highlight_function=lambda feature: {
            "color": "#000000",
            "weight": 2,
            "fillOpacity": 0.85,
        },
        tooltip=folium.Tooltip(tooltip_html, sticky=True),
        popup=folium.Popup(popup_html, max_width=380),
    ).add_to(grid_fg)

grid_fg.add_to(m)

# ===== 11) Administrative district boundaries =====
folium.GeoJson(
    town_wgs84[["geometry"]].to_json(),
    name="Taipei administrative district boundary",
    style_function=lambda feature: {
        "color": "#111111",
        "weight": 1.6,
        "fillColor": "transparent",
        "fillOpacity": 0,
    },
    interactive=False,
    show=True,
).add_to(m)

# ===== 12) Legend =====
cmap.add_to(m)

folium.LayerControl(collapsed=False).add_to(m)

map_path = OUTPUT_DIR / "Taipei_grid_travel_time_increase_rate_map.html"
m.save(map_path)

print("Saved:", map_path)

m

## 路網密度、地形條件與旅行時間增加比較

此步驟在「降雨量均相同」的情境下，比較不同 grid 的道路旅行時間增加情形。由於所有網格使用相同雨量，因此空間差異主要來自道路路網密度、道路型態組成、地形條件，以及原本的淹水風險特性。

分析會先將每個 grid 內的 road segment 彙整，計算道路總長度、道路密度、災前平均旅行時間、災後平均旅行時間與旅行時間增加率。接著再結合 `Taipei_grid_full_risk.geojson` 中的高程、坡度與風險分數，用來比較高密度 / 低密度路網，以及山區 / 平地 grid 的差異。

### 分類標準

| 分類項目 | 判斷標準 | 說明 |
|---|---|---|
| 路網密度 | `road_density_km_per_km2 = 道路總長度 km / grid 面積 km²` | 衡量每個 grid 內道路分布密集程度 |
| 高路網密度 | `road_density_km_per_km2 >= 全市有道路 grid 的中位數` | 路網密度高於或等於中位數 |
| 低路網密度 | `road_density_km_per_km2 < 全市有道路 grid 的中位數` | 路網密度低於中位數 |
| 山區或高坡 | `mean_elevation >= 100 m` 或 `slope >= 15` | 高程較高或坡度較大的 grid |
| 平地或低坡 | `mean_elevation < 100 m` 且 `slope < 15` | 高程與坡度皆較低的 grid |

### 主要輸出指標

| 指標 | 說明 |
|---|---|
| `avg_travel_time_increase_rate_pct` | grid 內道路災後平均旅行時間增加率 |
| `total_travel_time_increase_min` | grid 內所有道路總旅行時間增加分鐘數 |
| `travel_time_increase_per_km2_min` | 每平方公里旅行時間增加量 |
| `road_density_km_per_km2` | 每平方公里道路長度 |
| `closed_road_ratio_pct` | grid 內封閉道路比例 |

此分析可用來判斷在相同降雨條件下，路網密度較高或地形條件較不利的區域，是否會產生更明顯的道路旅行時間增加。

In [ ]:
import numpy as np
import pandas as pd
import geopandas as gpd
import folium
import branca.colormap as cm

from pathlib import Path

OUTPUT_DIR = Path("output")
TARGET_CRS = "EPSG:3826"

# ============================================================
# Grid comparison: road density, terrain, travel time increase
# ============================================================

# ===== 1) Read data =====
if "road_travel" not in globals():
    road_travel = gpd.read_file(
        OUTPUT_DIR / "road_grid_travel_time_pre_post.geojson"
    ).to_crs(TARGET_CRS)

grid = gpd.read_file(
    OUTPUT_DIR / "Taipei_grid_full_risk.geojson"
).to_crs(TARGET_CRS)

grid_base = grid[
    [
        "grid_id",
        "mean_elevation",
        "slope",
        "grid_area_m2",
        "geometry",
    ]
].copy()

road_grid = road_travel.copy()

# ===== 2) Clean fields =====
numeric_cols = [
    "segment_length_m",
    "pre_travel_time_sec",
    "post_travel_time_sec",
    "travel_time_increase_sec",
    "travel_time_increase_rate_pct",
]

for col in numeric_cols:
    road_grid[col] = pd.to_numeric(road_grid[col], errors="coerce")

road_grid["is_closed_post"] = (
    road_grid["is_closed_post"]
    .fillna(False)
    .astype(str)
    .str.lower()
    .isin(["true", "1", "yes"])
)

# ===== 3) Road density by grid =====
road_length_stats = (
    road_grid
    .groupby("grid_id", as_index=False)
    .agg(
        road_segment_count=("road_grid_id", "count"),
        total_road_length_m=("segment_length_m", "sum"),
        closed_road_count=("is_closed_post", "sum"),
    )
)

# ===== 4) Travel time increase by grid =====
# 封閉道路沒有合理的旅行時間，所以旅行時間增加率先用「仍可通行道路」計算
open_roads = road_grid[
    (~road_grid["is_closed_post"])
    & road_grid["pre_travel_time_sec"].notna()
    & road_grid["post_travel_time_sec"].notna()
    & (road_grid["pre_travel_time_sec"] > 0)
].copy()

travel_stats = (
    open_roads
    .groupby("grid_id", as_index=False)
    .agg(
        open_road_count=("road_grid_id", "count"),
        pre_travel_time_sum_sec=("pre_travel_time_sec", "sum"),
        post_travel_time_sum_sec=("post_travel_time_sec", "sum"),
        pre_avg_travel_time_sec=("pre_travel_time_sec", "mean"),
        post_avg_travel_time_sec=("post_travel_time_sec", "mean"),
    )
)

travel_stats["total_travel_time_increase_min"] = (
    travel_stats["post_travel_time_sum_sec"]
    - travel_stats["pre_travel_time_sum_sec"]
) / 60

travel_stats["avg_travel_time_increase_rate_pct"] = np.where(
    travel_stats["pre_travel_time_sum_sec"] > 0,
    (
        travel_stats["post_travel_time_sum_sec"]
        / travel_stats["pre_travel_time_sum_sec"]
        - 1
    ) * 100,
    np.nan,
)

travel_stats["pre_avg_travel_time_min"] = (
    travel_stats["pre_avg_travel_time_sec"] / 60
)

travel_stats["post_avg_travel_time_min"] = (
    travel_stats["post_avg_travel_time_sec"] / 60
)

# ===== 5) Join back to grid =====
grid_compare = grid_base.merge(
    road_length_stats,
    on="grid_id",
    how="left",
)

grid_compare = grid_compare.merge(
    travel_stats,
    on="grid_id",
    how="left",
)

fill_zero_cols = [
    "road_segment_count",
    "total_road_length_m",
    "closed_road_count",
    "open_road_count",
    "total_travel_time_increase_min",
]

for col in fill_zero_cols:
    grid_compare[col] = grid_compare[col].fillna(0)

grid_compare["grid_area_km2"] = grid_compare["grid_area_m2"] / 1_000_000

grid_compare["road_density_km_per_km2"] = np.where(
    grid_compare["grid_area_km2"] > 0,
    grid_compare["total_road_length_m"] / 1000 / grid_compare["grid_area_km2"],
    np.nan,
)

grid_compare["travel_time_increase_per_km2_min"] = np.where(
    grid_compare["grid_area_km2"] > 0,
    grid_compare["total_travel_time_increase_min"] / grid_compare["grid_area_km2"],
    np.nan,
)

grid_compare["closed_road_ratio_pct"] = np.where(
    grid_compare["road_segment_count"] > 0,
    grid_compare["closed_road_count"] / grid_compare["road_segment_count"] * 100,
    0,
)

# ===== 6) Classify road density and terrain =====
has_road = grid_compare["total_road_length_m"] > 0
density_median = grid_compare.loc[has_road, "road_density_km_per_km2"].median()

grid_compare["road_density_class"] = "無道路"
grid_compare.loc[
    has_road & (grid_compare["road_density_km_per_km2"] < density_median),
    "road_density_class",
] = "低路網密度"

grid_compare.loc[
    has_road & (grid_compare["road_density_km_per_km2"] >= density_median),
    "road_density_class",
] = "高路網密度"

# 山區 / 平地分類，可依你的研究定義調整門檻
grid_compare["terrain_class"] = np.where(
    (grid_compare["mean_elevation"] >= 100)
    | (grid_compare["slope"] >= 15),
    "山區或高坡",
    "平地或低坡",
)

# ===== 7) Summary comparison table =====
comparison_summary = (
    grid_compare[has_road]
    .groupby(["terrain_class", "road_density_class"], as_index=False)
    .agg(
        grid_count=("grid_id", "count"),
        mean_road_density_km_per_km2=("road_density_km_per_km2", "mean"),
        mean_pre_avg_travel_time_min=("pre_avg_travel_time_min", "mean"),
        mean_post_avg_travel_time_min=("post_avg_travel_time_min", "mean"),
        mean_increase_rate_pct=("avg_travel_time_increase_rate_pct", "mean"),
        mean_total_increase_min=("total_travel_time_increase_min", "mean"),
        mean_increase_per_km2_min=("travel_time_increase_per_km2_min", "mean"),
        mean_closed_road_ratio_pct=("closed_road_ratio_pct", "mean"),
    )
)

display(comparison_summary)

# ===== 8) Save outputs =====
grid_compare.to_file(
    OUTPUT_DIR / "Taipei_grid_density_terrain_travel_time_comparison.geojson",
    driver="GeoJSON",
    encoding="utf-8",
)

grid_compare.drop(columns="geometry").to_csv(
    OUTPUT_DIR / "Taipei_grid_density_terrain_travel_time_comparison.csv",
    index=False,
    encoding="utf-8-sig",
)

comparison_summary.to_csv(
    OUTPUT_DIR / "Taipei_density_terrain_comparison_summary.csv",
    index=False,
    encoding="utf-8-sig",
)

print("Saved grid comparison outputs.")